# Video Segmentation Methods Comparison for 3D Room Reconstruction

This notebook compares different video segmentation approaches to determine the best method for our 3D room reconstruction thesis project.

## Objectives:
1. **Test multiple segmentation approaches** - Compare traditional CV methods vs. modern deep learning approaches
2. **Evaluate performance** - Speed, accuracy, and memory usage for each method
3. **Assess applicability** - How well each method works for room/furniture detection
4. **Make informed decision** - Choose the optimal approach for our pipeline

## Methods to Compare:
- **Traditional CV**: OpenCV-based methods (watershed, background subtraction)
- **Deep Learning**: Segment Anything (SAM), Mask R-CNN, DeepLab, U-Net
- **Video-specific**: Track Anything, DETR for video
- **Real-time**: MobileNet-based segmentation

---

In [ ]:
# Import Required Libraries
import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Computer Vision
import cv2
from PIL import Image

# Deep Learning
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models.segmentation import deeplabv3_resnet50, fcn_resnet50

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Progress bars
from tqdm.notebook import tqdm

print("📚 Libraries imported successfully!")
print(f"🐍 Python version: {sys.version}")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"💻 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

## 1. Setup and Configuration

Let's set up our workspace and define helper functions for the comparison study.

In [ ]:
# Configuration and Helper Functions
class SegmentationConfig:
    """Configuration for segmentation experiments"""
    def __init__(self):
        # Paths
        self.data_dir = Path("../data")
        self.raw_data_dir = self.data_dir / "raw"
        self.processed_data_dir = self.data_dir / "processed"
        self.results_dir = Path("../experiments") / "segmentation_comparison"
        
        # Create directories
        self.raw_data_dir.mkdir(parents=True, exist_ok=True)
        self.processed_data_dir.mkdir(parents=True, exist_ok=True)
        self.results_dir.mkdir(parents=True, exist_ok=True)
        
        # Video processing parameters
        self.frame_skip = 5  # Process every 5th frame for speed
        self.max_frames = 50  # Maximum frames to process per video
        self.target_size = (512, 512)  # Resize frames for consistency
        
        # Device configuration
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"🖥️  Using device: {self.device}")

config = SegmentationConfig()

def preprocess_frame(frame, target_size=(512, 512)):
    """Preprocess frame for segmentation models"""
    # Resize frame
    frame_resized = cv2.resize(frame, target_size)
    
    # Convert BGR to RGB
    frame_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
    
    return frame_rgb

def calculate_metrics(pred_mask, gt_mask=None):
    """Calculate segmentation metrics"""
    metrics = {}
    
    # Basic statistics
    metrics['num_segments'] = len(np.unique(pred_mask))
    metrics['mask_coverage'] = np.sum(pred_mask > 0) / pred_mask.size
    
    # If ground truth is available
    if gt_mask is not None:
        # IoU calculation
        intersection = np.logical_and(pred_mask, gt_mask)
        union = np.logical_or(pred_mask, gt_mask)
        metrics['iou'] = np.sum(intersection) / np.sum(union)
        
        # Pixel accuracy
        metrics['pixel_accuracy'] = np.sum(pred_mask == gt_mask) / gt_mask.size
    
    return metrics

def visualize_segmentation(original, mask, title="Segmentation Result"):
    """Visualize segmentation results"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original image
    axes[0].imshow(original)
    axes[0].set_title("Original")
    axes[0].axis('off')
    
    # Segmentation mask
    axes[1].imshow(mask, cmap='viridis')
    axes[1].set_title("Segmentation Mask")
    axes[1].axis('off')
    
    # Overlay
    overlay = original.copy()
    colored_mask = plt.cm.viridis(mask / mask.max())[:, :, :3]
    overlay = 0.7 * overlay + 0.3 * (colored_mask * 255).astype(np.uint8)
    axes[2].imshow(overlay.astype(np.uint8))
    axes[2].set_title("Overlay")
    axes[2].axis('off')
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

print("✅ Configuration and helper functions ready!")

## 2. Load Sample Video Data

For testing, we'll use sample images and videos. If you don't have room videos yet, we can create synthetic test data or download sample videos.

In [ ]:
# Sample Data Preparation
def create_synthetic_room_scene(size=(512, 512)):
    """Create a synthetic room scene for testing"""
    image = np.zeros((*size, 3), dtype=np.uint8)
    
    # Floor (gray)
    image[int(size[0]*0.6):, :] = [80, 80, 80]
    
    # Wall (light gray)
    image[:int(size[0]*0.6), :] = [150, 150, 150]
    
    # Add some furniture-like rectangles
    # Sofa (brown)
    image[int(size[0]*0.5):int(size[0]*0.7), int(size[1]*0.1):int(size[1]*0.4)] = [139, 69, 19]
    
    # Table (wooden color)
    image[int(size[0]*0.6):int(size[0]*0.65), int(size[1]*0.5):int(size[1]*0.8)] = [160, 82, 45]
    
    # TV (black)
    image[int(size[0]*0.2):int(size[0]*0.4), int(size[1]*0.3):int(size[1]*0.7)] = [20, 20, 20]
    
    return image

def extract_frames_from_video(video_path, max_frames=50, frame_skip=5):
    """Extract frames from video file"""
    if not os.path.exists(video_path):
        print(f"⚠️  Video file not found: {video_path}")
        return []
    
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_count = 0
    
    while len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_count % frame_skip == 0:
            frame_rgb = preprocess_frame(frame)
            frames.append(frame_rgb)
        
        frame_count += 1
    
    cap.release()
    print(f"📹 Extracted {len(frames)} frames from video")
    return frames

# Try to load real video data or create synthetic data
test_frames = []
video_path = config.raw_data_dir / "sample_room.mp4"

if video_path.exists():
    print("📹 Loading frames from video...")
    test_frames = extract_frames_from_video(str(video_path))
else:
    print("🎨 Creating synthetic test data...")
    # Create multiple synthetic scenes with variations
    for i in range(5):
        synthetic_scene = create_synthetic_room_scene()
        # Add some noise and variations
        noise = np.random.randint(0, 30, synthetic_scene.shape, dtype=np.uint8)
        synthetic_scene = np.clip(synthetic_scene.astype(int) + noise, 0, 255).astype(np.uint8)
        test_frames.append(synthetic_scene)

print(f"✅ Prepared {len(test_frames)} test frames")

# Visualize first few frames
if test_frames:
    fig, axes = plt.subplots(1, min(3, len(test_frames)), figsize=(15, 5))
    if len(test_frames) == 1:
        axes = [axes]
    
    for i, frame in enumerate(test_frames[:3]):
        if i < len(axes):
            axes[i].imshow(frame)
            axes[i].set_title(f"Test Frame {i+1}")
            axes[i].axis('off')
    
    plt.suptitle("Sample Test Frames")
    plt.tight_layout()
    plt.show()

## 3. Traditional Computer Vision Segmentation

Let's start with traditional OpenCV-based segmentation methods to establish a baseline.

In [ ]:
# Traditional CV Segmentation Methods

class TraditionalSegmentation:
    """Traditional computer vision segmentation methods"""
    
    def __init__(self):
        self.results = {}
        
    def kmeans_segmentation(self, image, k=5):
        """K-means clustering for color-based segmentation"""
        start_time = time.time()
        
        # Reshape image to 1D array of pixels
        data = image.reshape((-1, 3)).astype(np.float32)
        
        # Apply K-means
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
        _, labels, centers = cv2.kmeans(data, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
        
        # Reshape back to image
        segmented = labels.reshape(image.shape[:2])
        
        processing_time = time.time() - start_time
        return segmented, processing_time
    
    def watershed_segmentation(self, image):
        """Watershed algorithm for segmentation"""
        start_time = time.time()
        
        # Convert to grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        
        # Threshold
        ret, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        
        # Noise removal
        kernel = np.ones((3, 3), np.uint8)
        opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
        
        # Sure background area
        sure_bg = cv2.dilate(opening, kernel, iterations=3)
        
        # Finding sure foreground area
        dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
        ret, sure_fg = cv2.threshold(dist_transform, 0.7 * dist_transform.max(), 255, 0)
        
        # Finding unknown region
        sure_fg = np.uint8(sure_fg)
        unknown = cv2.subtract(sure_bg, sure_fg)
        
        # Marker labelling
        ret, markers = cv2.connectedComponents(sure_fg)
        markers = markers + 1
        markers[unknown == 255] = 0
        
        # Apply watershed
        img_copy = image.copy()
        markers = cv2.watershed(img_copy, markers)
        
        processing_time = time.time() - start_time
        return markers, processing_time
    
    def grabcut_segmentation(self, image):
        """GrabCut algorithm for foreground/background separation"""
        start_time = time.time()
        
        # Create mask
        mask = np.zeros(image.shape[:2], np.uint8)
        
        # Create rectangle for probable foreground (center region)
        height, width = image.shape[:2]
        rect = (width//4, height//4, width//2, height//2)
        
        # Background and foreground models
        bgdModel = np.zeros((1, 65), np.float64)
        fgdModel = np.zeros((1, 65), np.float64)
        
        # Apply GrabCut
        cv2.grabCut(image, mask, rect, bgdModel, fgdModel, 5, cv2.GC_INIT_WITH_RECT)
        
        # Create final mask
        mask2 = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')
        
        processing_time = time.time() - start_time
        return mask2, processing_time

# Initialize traditional segmentation
traditional_seg = TraditionalSegmentation()

# Test on first frame
if test_frames:
    test_frame = test_frames[0]
    print("🔬 Testing traditional CV segmentation methods...")
    
    # K-means segmentation
    kmeans_result, kmeans_time = traditional_seg.kmeans_segmentation(test_frame, k=6)
    print(f"   K-means: {kmeans_time:.3f}s")
    
    # Watershed segmentation  
    watershed_result, watershed_time = traditional_seg.watershed_segmentation(test_frame)
    print(f"   Watershed: {watershed_time:.3f}s")
    
    # GrabCut segmentation
    grabcut_result, grabcut_time = traditional_seg.grabcut_segmentation(test_frame)
    print(f"   GrabCut: {grabcut_time:.3f}s")
    
    # Visualize results
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    axes[0, 0].imshow(test_frame)
    axes[0, 0].set_title("Original")
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(kmeans_result, cmap='viridis')
    axes[0, 1].set_title(f"K-means ({kmeans_time:.3f}s)")
    axes[0, 1].axis('off')
    
    axes[1, 0].imshow(watershed_result, cmap='viridis')
    axes[1, 0].set_title(f"Watershed ({watershed_time:.3f}s)")
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(grabcut_result, cmap='gray')
    axes[1, 1].set_title(f"GrabCut ({grabcut_time:.3f}s)")
    axes[1, 1].axis('off')
    
    plt.suptitle("Traditional CV Segmentation Results")
    plt.tight_layout()
    plt.show()
    
    # Store results for comparison
    traditional_seg.results = {
        'kmeans': {'mask': kmeans_result, 'time': kmeans_time, 'metrics': calculate_metrics(kmeans_result)},
        'watershed': {'mask': watershed_result, 'time': watershed_time, 'metrics': calculate_metrics(watershed_result)},
        'grabcut': {'mask': grabcut_result, 'time': grabcut_time, 'metrics': calculate_metrics(grabcut_result)}
    }

## 4. Deep Learning Segmentation Models

Now let's test state-of-the-art deep learning models for semantic segmentation.

In [ ]:
# Deep Learning Segmentation Models

class DeepLearningSegmentation:
    """Deep learning-based segmentation methods"""
    
    def __init__(self, device='cpu'):
        self.device = device
        self.models = {}
        self.results = {}
        
    def load_deeplabv3(self):
        """Load pre-trained DeepLabV3 model"""
        print("🔄 Loading DeepLabV3...")
        try:
            model = deeplabv3_resnet50(pretrained=True)
            model.eval()
            model.to(self.device)
            self.models['deeplabv3'] = model
            print("✅ DeepLabV3 loaded successfully")
            return True
        except Exception as e:
            print(f"❌ Failed to load DeepLabV3: {e}")
            return False
    
    def load_fcn(self):
        """Load pre-trained FCN model"""
        print("🔄 Loading FCN...")
        try:
            model = fcn_resnet50(pretrained=True)
            model.eval()
            model.to(self.device)
            self.models['fcn'] = model
            print("✅ FCN loaded successfully")
            return True
        except Exception as e:
            print(f"❌ Failed to load FCN: {e}")
            return False
    
    def preprocess_for_model(self, image):
        """Preprocess image for PyTorch models"""
        # Convert to tensor and normalize
        transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        tensor = transform(image).unsqueeze(0).to(self.device)
        return tensor
    
    def segment_with_model(self, image, model_name):
        """Segment image using specified model"""
        if model_name not in self.models:
            print(f"❌ Model {model_name} not loaded")
            return None, 0
        
        start_time = time.time()
        
        # Preprocess image
        input_tensor = self.preprocess_for_model(image)
        
        # Run inference
        with torch.no_grad():
            output = self.models[model_name](input_tensor)['out'][0]
            output_predictions = output.argmax(0).cpu().numpy()
        
        processing_time = time.time() - start_time
        return output_predictions, processing_time
    
    def segment_all_models(self, image):
        """Run segmentation with all loaded models"""
        results = {}
        
        for model_name in self.models.keys():
            print(f"🔬 Running {model_name}...")
            mask, time_taken = self.segment_with_model(image, model_name)
            
            if mask is not None:
                metrics = calculate_metrics(mask)
                results[model_name] = {
                    'mask': mask,
                    'time': time_taken,
                    'metrics': metrics
                }
                print(f"   ✅ {model_name}: {time_taken:.3f}s, {metrics['num_segments']} segments")
            else:
                print(f"   ❌ {model_name}: Failed")
        
        return results

# Initialize deep learning segmentation
dl_seg = DeepLearningSegmentation(device=config.device)

# Load models
print("📥 Loading deep learning models...")
dl_seg.load_deeplabv3()
dl_seg.load_fcn()

# Note: For Segment Anything (SAM), we'd need to install it separately:
# pip install git+https://github.com/facebookresearch/segment-anything.git

def load_segment_anything():
    """Load Segment Anything Model (if available)"""
    try:
        # This would require: pip install segment-anything
        # from segment_anything import SamPredictor, sam_model_registry
        # model = sam_model_registry["vit_b"](checkpoint="path/to/sam_vit_b.pth")
        # predictor = SamPredictor(model)
        print("ℹ️  Segment Anything not available - install with:")
        print("   pip install git+https://github.com/facebookresearch/segment-anything.git")
        return None
    except ImportError:
        print("ℹ️  Segment Anything not installed")
        return None

sam_model = load_segment_anything()

print(f"🎯 Loaded {len(dl_seg.models)} deep learning models")

In [ ]:
# Test Deep Learning Models
if test_frames and dl_seg.models:
    test_frame = test_frames[0]
    print("🧠 Testing deep learning segmentation models...")
    
    # Run all models
    dl_results = dl_seg.segment_all_models(test_frame)
    
    # Visualize results
    num_models = len(dl_results)
    if num_models > 0:
        fig, axes = plt.subplots(1, num_models + 1, figsize=(5 * (num_models + 1), 5))
        if num_models == 1:
            axes = [axes] if isinstance(axes, np.ndarray) else [axes]
        
        # Original image
        axes[0].imshow(test_frame)
        axes[0].set_title("Original")
        axes[0].axis('off')
        
        # Model results
        for i, (model_name, result) in enumerate(dl_results.items()):
            axes[i + 1].imshow(result['mask'], cmap='viridis')
            axes[i + 1].set_title(f"{model_name.upper()}\n({result['time']:.3f}s)")
            axes[i + 1].axis('off')
        
        plt.suptitle("Deep Learning Segmentation Results")
        plt.tight_layout()
        plt.show()
        
        # Store results
        dl_seg.results = dl_results
    
else:
    print("⚠️  No test frames or models available")

## 5. Performance Comparison and Analysis

Let's compare all methods across different metrics: speed, quality, and applicability to room segmentation.

In [ ]:
# Performance Comparison and Analysis

def compile_all_results():
    """Compile results from all segmentation methods"""
    all_results = {}
    
    # Traditional CV results
    if hasattr(traditional_seg, 'results'):
        for method, result in traditional_seg.results.items():
            all_results[f"CV_{method}"] = result
    
    # Deep learning results  
    if hasattr(dl_seg, 'results'):
        for method, result in dl_seg.results.items():
            all_results[f"DL_{method}"] = result
    
    return all_results

# Compile all results
all_results = compile_all_results()

if all_results:
    print(f"📊 Comparing {len(all_results)} segmentation methods...")
    
    # Create comparison DataFrame
    comparison_data = []
    for method, result in all_results.items():
        row = {
            'Method': method,
            'Type': 'Traditional CV' if method.startswith('CV_') else 'Deep Learning',
            'Processing Time (s)': result['time'],
            'Number of Segments': result['metrics']['num_segments'],
            'Mask Coverage': result['metrics']['mask_coverage'],
            'Speed (FPS)': 1.0 / result['time'] if result['time'] > 0 else float('inf')
        }
        comparison_data.append(row)
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Display comparison table
    print("\n📋 Method Comparison Summary:")
    print(comparison_df.round(3))
    
    # Create visualization
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Processing Time', 'Speed (FPS)', 'Number of Segments', 'Mask Coverage'),
        specs=[[{"type": "bar"}, {"type": "bar"}],
               [{"type": "bar"}, {"type": "bar"}]]
    )
    
    # Processing time
    fig.add_trace(
        go.Bar(x=comparison_df['Method'], y=comparison_df['Processing Time (s)'], 
               marker_color=['lightcoral' if 'CV_' in x else 'lightblue' for x in comparison_df['Method']]),
        row=1, col=1
    )
    
    # Speed (FPS)
    fig.add_trace(
        go.Bar(x=comparison_df['Method'], y=comparison_df['Speed (FPS)'], 
               marker_color=['lightcoral' if 'CV_' in x else 'lightblue' for x in comparison_df['Method']]),
        row=1, col=2
    )
    
    # Number of segments
    fig.add_trace(
        go.Bar(x=comparison_df['Method'], y=comparison_df['Number of Segments'], 
               marker_color=['lightcoral' if 'CV_' in x else 'lightblue' for x in comparison_df['Method']]),
        row=2, col=1
    )
    
    # Mask coverage
    fig.add_trace(
        go.Bar(x=comparison_df['Method'], y=comparison_df['Mask Coverage'], 
               marker_color=['lightcoral' if 'CV_' in x else 'lightblue' for x in comparison_df['Method']]),
        row=2, col=2
    )
    
    fig.update_layout(
        title_text="Segmentation Methods Performance Comparison",
        showlegend=False,
        height=600
    )
    
    # Update x-axis labels to be rotated
    for i in range(1, 3):
        for j in range(1, 3):
            fig.update_xaxes(tickangle=45, row=i, col=j)
    
    fig.show()
    
    # Save comparison results
    comparison_df.to_csv(config.results_dir / "segmentation_comparison.csv", index=False)
    print(f"\n💾 Comparison results saved to: {config.results_dir / 'segmentation_comparison.csv'}")

else:
    print("⚠️  No results to compare. Run the segmentation methods first.")

## 6. Recommendations and Next Steps

Based on the comparison, let's analyze which approach is best suited for 3D room reconstruction.

In [ ]:
# Analysis and Recommendations

def analyze_results(results_df):
    """Analyze results and provide recommendations"""
    if results_df is None or len(results_df) == 0:
        return "No results to analyze"
    
    analysis = {
        'fastest_method': results_df.loc[results_df['Processing Time (s)'].idxmin(), 'Method'],
        'highest_fps': results_df.loc[results_df['Speed (FPS)'].idxmax(), 'Method'], 
        'most_segments': results_df.loc[results_df['Number of Segments'].idxmax(), 'Method'],
        'best_coverage': results_df.loc[results_df['Mask Coverage'].idxmax(), 'Method']
    }
    
    return analysis

if 'comparison_df' in locals():
    analysis = analyze_results(comparison_df)
    
    print("🎯 ANALYSIS SUMMARY")
    print("=" * 50)
    print(f"⚡ Fastest Processing: {analysis['fastest_method']}")
    print(f"🏃 Highest FPS: {analysis['highest_fps']}")
    print(f"🔍 Most Detailed: {analysis['most_segments']}")
    print(f"📊 Best Coverage: {analysis['best_coverage']}")
    
    print("\n🏗️ RECOMMENDATIONS FOR 3D ROOM RECONSTRUCTION")
    print("=" * 50)
    
    recommendations = [
        {
            "Method": "Segment Anything (SAM)",
            "Pros": ["State-of-the-art accuracy", "Can segment anything", "Good for furniture detection"],
            "Cons": ["Requires installation", "Computationally intensive"],
            "Use Case": "High-quality segmentation with manual prompts"
        },
        {
            "Method": "DeepLabV3",
            "Pros": ["Pre-trained", "Good semantic understanding", "Real-time capable"],
            "Cons": ["Fixed categories", "May miss custom furniture"],
            "Use Case": "Real-time processing with good accuracy"
        },
        {
            "Method": "K-means + Watershed",
            "Pros": ["Fast", "No GPU required", "Good for color-based segmentation"],
            "Cons": ["Limited semantic understanding", "Sensitive to lighting"],
            "Use Case": "Fast preprocessing or fallback method"
        }
    ]
    
    for i, rec in enumerate(recommendations, 1):
        print(f"\n{i}. {rec['Method']}")
        print(f"   ✅ Pros: {', '.join(rec['Pros'])}")
        print(f"   ❌ Cons: {', '.join(rec['Cons'])}")
        print(f"   🎯 Use Case: {rec['Use Case']}")
    
    print("\n🚀 NEXT STEPS")
    print("=" * 30)
    next_steps = [
        "Install and test Segment Anything (SAM) model",
        "Test on real room video data",
        "Implement room-specific object detection",
        "Integrate best method into 3D reconstruction pipeline",
        "Benchmark on larger dataset"
    ]
    
    for i, step in enumerate(next_steps, 1):
        print(f"{i}. {step}")
    
    print("\n💡 THESIS CONTRIBUTION IDEAS")
    print("=" * 30)
    contributions = [
        "Compare segmentation methods specifically for indoor scenes",
        "Develop hybrid approach combining multiple methods",
        "Create room-specific segmentation dataset",
        "Optimize segmentation for 3D reconstruction pipeline",
        "Study temporal consistency in video segmentation"
    ]
    
    for i, contrib in enumerate(contributions, 1):
        print(f"{i}. {contrib}")

else:
    print("⚠️  Run the comparison first to see analysis")

## 7. Installation Instructions for Advanced Methods

Run these commands to install additional segmentation models:

In [ ]:
# Installation commands for advanced segmentation methods

installation_commands = {
    "Segment Anything (SAM)": [
        "pip install git+https://github.com/facebookresearch/segment-anything.git",
        "# Download model weights:",
        "# wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
    ],
    
    "Detectron2 (Mask R-CNN)": [
        "# For CUDA 11.8:",
        "pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu118/torch2.0/index.html",
        "# For CPU only:",
        "pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cpu/torch2.0/index.html"
    ],
    
    "Track Anything (Video Segmentation)": [
        "pip install git+https://github.com/jiaaro/pydub.git",
        "pip install git+https://github.com/gaoxiangnumber1/Track-Anything.git"
    ],
    
    "MMSegmentation": [
        "pip install -U openmim", 
        "mim install mmengine",
        "mim install mmcv",
        "pip install mmsegmentation"
    ]
}

print("🛠️  INSTALLATION COMMANDS")
print("=" * 50)

for method, commands in installation_commands.items():
    print(f"\n📦 {method}:")
    for cmd in commands:
        if cmd.startswith("#"):
            print(f"   {cmd}")
        else:
            print(f"   $ {cmd}")

print("\n⚠️  IMPORTANT NOTES:")
print("- Install PyTorch with CUDA support first if you have a GPU")
print("- Some models require downloading large weight files (~2GB)")
print("- Segment Anything performs best but requires more setup")
print("- For thesis work, I recommend starting with DeepLabV3 and then adding SAM")

print("\n🎯 RECOMMENDED INSTALLATION ORDER:")
print("1. Start with built-in PyTorch models (DeepLabV3, FCN) - already available")
print("2. Install Detectron2 for Mask R-CNN") 
print("3. Install Segment Anything for state-of-the-art results")
print("4. Consider Track Anything for video-specific features")

# Create installation script
install_script = """#!/bin/bash
# Segmentation Models Installation Script

echo "Installing segmentation models for thesis project..."

# Basic requirements (should already be installed)
pip install torch torchvision opencv-python pillow numpy

# Detectron2 (adjust CUDA version as needed)
pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu118/torch2.0/index.html

# Segment Anything
pip install git+https://github.com/facebookresearch/segment-anything.git

# Download SAM model weights
mkdir -p ../data/models
cd ../data/models
wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
cd ../../notebooks

echo "Installation complete! Run the notebook again to test all methods."
"""

# Save installation script
script_path = config.results_dir / "install_segmentation_models.sh"
with open(script_path, 'w') as f:
    f.write(install_script)

print(f"\n💾 Installation script saved to: {script_path}")
print("Run with: bash install_segmentation_models.sh")